In [41]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

In [42]:
movies = pd.read_csv('data/movies.csv')
credits = pd.read_csv('data/credits.csv')

movies = movies.merge(credits, on='title')

In [43]:
movies = movies[[
    'movie_id',
    'title',
    'overview',
    'genres',
    'keywords',
    'cast',
    'crew'
]]

In [44]:
movies.dropna(inplace=True)

In [45]:
def convert(text):
    L = []

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [46]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [47]:
def convert_cast(text):

    L = []
    counter = 0

    for i in ast.literal_eval(text):

        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [48]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [49]:
def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [50]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [51]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [52]:
movies['genres'] = movies['genres'].apply(
    lambda x:[i.replace(' ','') for i in x]
)

movies['keywords'] = movies['keywords'].apply(
    lambda x:[i.replace(' ','') for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x:[i.replace(' ','') for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x:[i.replace(' ','') for i in x]
)

In [53]:
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['crew']
)

In [54]:
new_df = movies[['movie_id','title','tags']]

In [55]:
new_df['tags'] = new_df['tags'].apply(lambda x:' '.join(x))

In [56]:
!pip install nltk

In [57]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):

    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return ' '.join(y)

In [58]:
new_df['tags'] = new_df['tags'].apply(stem)

In [59]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

In [60]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [61]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [62]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [63]:
recommend('Batman Begins')

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [64]:
import pickle

pickle.dump(new_df, open('../artifacts/movies.pkl','wb'))
pickle.dump(similarity, open('../artifacts/similarity.pkl','wb'))